# Transformers and Regulators

A `PowerTransformer` is described by its ends, one `PowerTransformerEnd` per winding with its ratings and
impedance. That is enough for a balanced model or a substation transformer. A tap changer on one end, with
a control, makes it regulate voltage. A `VoltageRegulator` is a step-voltage regulator, a single-phase
autotransformer with a tap changer.

To describe each physical tank of a distribution transformer, its lugs, core and windings, see
[Transformer tanks](3_7_transformer_tanks.ipynb).

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs. `new_feeder_model` makes an empty model to build into.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    BaseVoltageBuilder,
    PowerTransformerBuilder,
    RatioTapChangerBuilder,
    TapChangerControlBuilder,
    VoltageRegulatorBuilder,
)

network = new_feeder_model(
    authority="https://example.com/models", model="transformers", Feeder="fdr_demo"
)
container = network.container
base_voltage_builder = BaseVoltageBuilder(network, container)
for mRID, nominalVoltage in [
    ("bv_115kV", Qty(115, "kV")),
    ("bv_12_47kV", Qty(12.47, "kV")),
]:
    base_voltage_builder.create(mRID=mRID, name=mRID, nominalVoltage=nominalVoltage)

## A two-winding transformer

`add_connectivity` puts the transformer in its container. Each `add_power_transformer_end` then adds an
end with its Terminal on a node, named `<transformer>_<end number>` and `<transformer>_T<end number>`. End
1 is the highest-voltage winding.

`add_electrical` sets one end's ratings and impedance. The series impedance of a two-winding transformer
goes on end 1, referred to end 1's voltage. `add_short_circuit` sets its zero-sequence impedance.

In [2]:
xfmr_builder = PowerTransformerBuilder(network, container)
xfmr = xfmr_builder.create(mRID="xf_sub", name="substation")
xfmr_builder.add_connectivity(xfmr)
xfmr, _, high_end = xfmr_builder.add_power_transformer_end(
    xfmr, node="hv_bus", endNumber=1, connectionKind="D", BaseVoltage="bv_115kV"
)
xfmr, low_terminal, low_end = xfmr_builder.add_power_transformer_end(
    xfmr,
    node="lv_bus",
    endNumber=2,
    connectionKind="Yn",
    grounded=True,
    BaseVoltage="bv_12_47kV",
)
xfmr_builder.add_electrical(
    high_end,
    r=Qty(6.6, "ohm"),
    x=Qty(105, "ohm"),
    ratedS=Qty(10, "MVA"),
    ratedU=Qty(115, "kV"),
)
xfmr_builder.add_electrical(low_end, ratedS=Qty(10, "MVA"), ratedU=Qty(12.47, "kV"))
xfmr_builder.add_short_circuit(high_end, r0=Qty(6.6, "ohm"), x0=Qty(105, "ohm"))

for end in sorted(xfmr.PowerTransformerEnd, key=lambda end: end.endNumber):
    print(
        end.mRID,
        end.connectionKind.value,
        end.Terminal.mRID,
        f"{end.ratedS.to('MVA'):g} MVA",
        f"{end.ratedU.to('kV'):g} kV",
    )

xf_sub_1 D xf_sub_T1 10 MVA 115 kV
xf_sub_2 Yn xf_sub_T2 10 MVA 12.47 kV


A transformer with three or more windings gives the impedance between each pair of ends instead, with
`add_mesh_impedance(xfmr, FromTransformerEnd=..., ToTransformerEnd=..., r=..., x=...)`.

## A load tap changer

Voltage regulation takes two objects:

- a `TapChangerControl`, what to regulate: here the voltage at the transformer's low-voltage Terminal,
- a `RatioTapChanger` on the end it adjusts, with its step range and step size in percent.

This one gives ±10 % in 32 steps of 0.625 %. The control's target and deadband are in volts on the
primary side. Here the target is 124 V on a 120 V base, through a 60:1 potential transformer.

In [3]:
control_builder = TapChangerControlBuilder(network, container)
control = control_builder.create(mRID="xf_sub_ctrl")
control_builder.add_regulating_control(
    control,
    Terminal=low_terminal.mRID,
    mode="voltage",
    monitoredPhase="A",
    targetValue=124 * 60,
    targetDeadband=2 * 60,
    discrete=True,
    enabled=True,
)

tap_builder = RatioTapChangerBuilder(network, container)
tap_changer = tap_builder.create(mRID="xf_sub_ltc")
tap_builder.add_connectivity(tap_changer, TransformerEnd=low_end.mRID)
tap_builder.add_electrical(
    tap_changer, highStep=16, lowStep=-16, normalStep=0, stepVoltageIncrement=0.625
)
tap_builder.add_regulating_control(
    tap_changer,
    neutralStep=0,
    neutralU=Qty(12.47, "kV"),
    TapChangerControl=control.mRID,
    ltcFlag=True,
    controlEnabled=True,
    ptRatio=60,
    initialDelay=Qty(30, "s"),
    subsequentDelay=Qty(5, "s"),
)
tap_builder.add_ssh(tap_changer, step=2)

print(f"{low_end.RatioTapChanger.mRID} at step {tap_changer.step:g}")
print("controlled by", tap_changer.TapChangerControl.mRID)

xf_sub_ltc at step 2
controlled by xf_sub_ctrl


## A step-voltage regulator

A `VoltageRegulator` is an autotransformer. Each phase of a three-phase regulator bank is its own
single-phase regulator. `add_connectivity` sets the ANSI construction type. The regulator then gets:

- an `AutoTransformerEnd` with a Terminal on each side, source and load, carrying the regulator's phase,
- the lugs of its series winding: `S` (source), `L` (load) and `SL` (the common terminal),
- the winding itself, between the lugs, with its rating.

In a type A regulator the tap changer is on the load end, controlled from the load Terminal. This one is a
7620 V, 100 A, 76.2 kVA regulator on phase A.

In [4]:
regulator_builder = VoltageRegulatorBuilder(network, container)
regulator = regulator_builder.create(mRID="vr_1a", name="reg 1 phase A")
regulator_builder.add_connectivity(regulator, construction="typeA")

ends = {}
for endNumber, node in [(1, "reg_1_source"), (2, "reg_1_load")]:
    regulator, _, ends[endNumber] = regulator_builder.add_auto_transformer_end(
        regulator,
        node=node,
        endNumber=endNumber,
        phase="AN",
        BaseVoltage="bv_12_47kV",
    )
    regulator_builder.add_electrical(
        ends[endNumber], baseU=Qty(7.62, "kV"), ratedS=Qty(762, "kVA")
    )

regulator_builder.add_transformer_lug(
    regulator, designation="S", sequenceNumber=1, TransformerEnd=ends[1].mRID
)
regulator_builder.add_transformer_lug(
    regulator, designation="L", sequenceNumber=2, TransformerEnd=ends[2].mRID
)
regulator_builder.add_transformer_lug(regulator, designation="SL", sequenceNumber=3)
regulator, winding = regulator_builder.add_transformer_winding(
    regulator,
    FromTransformerEnd=ends[1].mRID,
    ToTransformerEnd=ends[2].mRID,
    FromTransformerLug="vr_1a_lug_s",
    ToTransformerLug="vr_1a_lug_sl",
    AutoTransformerLug="vr_1a_lug_l",
    ratedS=Qty(76.2, "kVA"),
    baseU=Qty(7.62, "kV"),
)

control = control_builder.create(mRID="vr_1a_ctrl")
control_builder.add_regulating_control(
    control,
    Terminal=ends[2].Terminal.mRID,
    mode="voltage",
    monitoredPhase="A",
    targetValue=7620,
    targetDeadband=127,
    discrete=True,
    enabled=True,
    lineDropCompensation=False,
)
tap_changer = tap_builder.create(mRID="vr_1a_tap")
tap_builder.add_connectivity(tap_changer, TransformerEnd=ends[2].mRID)
tap_builder.add_electrical(
    tap_changer, highStep=16, lowStep=-16, normalStep=0, stepVoltageIncrement=0.625
)
tap_builder.add_regulating_control(
    tap_changer,
    neutralStep=0,
    neutralU=Qty(7.62, "kV"),
    TapChangerControl=control.mRID,
    ltcFlag=True,
    controlEnabled=True,
    ptRatio=63.5,
    ctRating=100,
)
tap_builder.add_ssh(tap_changer, step=0)

print([(end.mRID, end.Terminal.ConnectivityNode.mRID) for end in ends.values()])
print(
    winding.mRID,
    winding.FromTransformerLug.designation,
    "-",
    winding.ToTransformerLug.designation,
)

[('vr_1a_end1', 'reg_1_source'), ('vr_1a_end2', 'reg_1_load')]
vr_1a_wdg S - SL


For a gang-operated three-phase bank, build one regulator per phase and give all three tap changers the
same `TapChangerControl`.